# DataMart Presupuestal Municipal — Documentación Completa

**Arquitectura de Data Warehouse / Data Mart Presupuestal Municipal**

Este notebook documenta el proceso de construcción e implementación del Data Mart Presupuestal Municipal utilizando Python, SQL Server y Power BI.

| Fase | Descripción | Herramienta |
|---|---|---|
| 1 | Construcción de la capa Gold | Python / Pandas |
| 2 | Diseño del modelo dimensional | Modelo Estrella |
| 3 | Implementación física del Data Mart | SQL Server |
| 4 | Desarrollo del proceso ETL | Python + SQLAlchemy |
| 5 | Integración analítica | Power BI |

---

# FASE 1 — Construcción de la Capa Gold

En esta fase se consolidó el dataset final a partir de la información previamente limpiada y transformada en las capas anteriores del pipeline. Se optimizó la estructura de datos para su explotación analítica y posterior integración con el Data Mart.

## Procesos realizados

| Proceso | Descripción |
|---|---|
| Consolidación final | Integración completa de datos limpios |
| Optimización analítica | Dataset preparado para consultas OLAP |
| Eliminación de columnas nulas | Reducción de almacenamiento innecesario |
| Exportación final | Generación del archivo Parquet |
| Integridad de datos | Validación final antes del Data Mart |

## Resultado Final

| Métrica | Resultado |
|---|---|
| Registros finales | 2,852,681 |
| Cobertura temporal | 2012 - 2026 |
| Municipalidades | 1,964 |
| Departamentos | 36 |
| Formato final | Parquet |

---



# FASE 2 — Diseño del Modelo Dimensional

En esta fase se diseñó el modelo dimensional del Data Mart utilizando arquitectura tipo estrella basada en la metodología Kimball. Se definieron las tablas dimensionales y la tabla de hechos necesarias para soportar análisis multidimensionales sobre la ejecución presupuestal municipal.

## Arquitectura Implementada

| Componente | Descripción |
|---|---|
| Tabla Fact | FACT_PRESUPUESTO |
| Dimensiones | 4 dimensiones |
| Modelo | Esquema Estrella |
| Arquitectura | Kimball |
| Orientación | OLAP |



## Modelo Dimensional

<div align="center">

<img src="modelo_estrella.png" width="900">

</div>


## Tabla de Hechos — FACT_PRESUPUESTO

La tabla central almacena las métricas financieras principales del sistema presupuestal municipal.

| Métrica | Descripción |
|---|---|
| MONTO_PIA | Presupuesto Inicial de Apertura |
| MONTO_PIM | Presupuesto Institucional Modificado |
| MONTO_RECAUDADO | Recaudación municipal |
| TASA_EJECUCION | Nivel de ejecución presupuestal |



## Tablas Dimensionales

### DIM_TIEMPO

Permite realizar análisis temporales y tendencias financieras.

| Campo | Descripción |
|---|---|
| anio | Año presupuestal |
| mes | Mes |
| trimestre | Trimestre |
| semestre | Semestre |
| bimestre | Bimestre |



### DIM_UBICACION

Contiene la estructura geográfica territorial.

| Campo | Descripción |
|---|---|
| UBIGEO | Código geográfico |
| departamento | Departamento |
| provincia | Provincia |
| distrito | Distrito |



### DIM_ENTIDAD

Describe las entidades municipales y ejecutoras.

| Campo | Descripción |
|---|---|
| SEC_EJEC | Código ejecutora |
| EJECUTORA_NOMBRE | Municipalidad |
| NIVEL_GOBIERNO | Nivel gubernamental |
| Tipomuni | Tipo de municipalidad |



### DIM_FINANCIERA

Estructura presupuestal financiera del SIAF.

| Campo | Descripción |
|---|---|
| FUENTE_FINANCIAMIENTO | Fuente financiera |
| RUBRO | Rubro presupuestal |
| GENERICA | Genérica |
| SUBGENERICA | Subgenérica |

---

# FASE 3 — Implementación Física en SQL Server

En esta fase se implementó físicamente el Data Mart en SQL Server. Se crearon las tablas dimensionales, la tabla de hechos, las relaciones mediante llaves foráneas y las restricciones de calidad de datos necesarias para garantizar integridad y rendimiento analítico.

## Componentes Implementados

| Componente | Implementación |
|---|---|
| Base de datos | DMT_PRESUPUESTO |
| Esquema | dmt |
| Tabla Fact | FACT_PRESUPUESTO |
| Dimensiones | 4 dimensiones |
| Llaves foráneas | Integridad referencial |
| Constraints | Validaciones de calidad |
| Columnstore | Optimización OLAP |

## Script SQL del Data Mart

<details>
<summary><strong>Ver Script SQL Completo</strong></summary>

```sql
CREATE DATABASE DMT_PRESUPUESTO;
GO

USE DMT_PRESUPUESTO;
GO

CREATE SCHEMA dmt;
GO

CREATE TABLE dmt.DIM_TIEMPO (
    id_tiempo INT IDENTITY(1,1) PRIMARY KEY,
    fecha_inicio_mes DATE NOT NULL,
    anio INT NOT NULL,
    mes INT NOT NULL,
    nombre_mes VARCHAR(20) NOT NULL,
    nombre_corto_mes VARCHAR(5) NOT NULL,
    trimestre VARCHAR(5) NOT NULL,
    semestre VARCHAR(5) NOT NULL,
    bimestre VARCHAR(5) NOT NULL
);
GO

CREATE TABLE dmt.DIM_UBICACION (
    id_ubicacion INT IDENTITY(1,1) PRIMARY KEY,
    UBIGEO VARCHAR(10) NOT NULL,
    departamento VARCHAR(100),
    provincia VARCHAR(100),
    distrito VARCHAR(100)
);
GO

CREATE TABLE dmt.DIM_ENTIDAD (
    id_entidad INT IDENTITY(1,1) PRIMARY KEY,
    SEC_EJEC VARCHAR(20) NOT NULL,
    EJECUTORA INT,
    EJECUTORA_NOMBRE VARCHAR(255),
    NIVEL_GOBIERNO VARCHAR(50),
    NIVEL_GOBIERNO_NOMBRE VARCHAR(150),
    DEPARTAMENTO_EJECUTORA_NOMBRE VARCHAR(150),
    PROVINCIA_EJECUTORA_NOMBRE VARCHAR(150),
    DISTRITO_EJECUTORA_NOMBRE VARCHAR(150),
    Tipomuni FLOAT
);
GO

CREATE TABLE dmt.DIM_FINANCIERA (
    id_financiera INT IDENTITY(1,1) PRIMARY KEY,
    FUENTE_FINANCIAMIENTO VARCHAR(50),
    FUENTE_FINANCIAMIENTO_NOMBRE VARCHAR(255),
    RUBRO VARCHAR(50),
    RUBRO_NOMBRE VARCHAR(255),
    GENERICA VARCHAR(50),
    GENERICA_NOMBRE VARCHAR(255),
    SUBGENERICA VARCHAR(50),
    SUBGENERICA_NOMBRE VARCHAR(255)
);
GO

SET IDENTITY_INSERT dmt.DIM_TIEMPO ON;

INSERT INTO dmt.DIM_TIEMPO (
    id_tiempo,
    fecha_inicio_mes,
    anio,
    mes,
    nombre_mes,
    nombre_corto_mes,
    trimestre,
    semestre,
    bimestre
)
VALUES (
    -1,
    '1900-01-01',
    0,
    0,
    'No Especificado',
    'N/E',
    'T0',
    'S0',
    'B0'
);

SET IDENTITY_INSERT dmt.DIM_TIEMPO OFF;

SET IDENTITY_INSERT dmt.DIM_UBICACION ON;

INSERT INTO dmt.DIM_UBICACION (
    id_ubicacion,
    UBIGEO,
    departamento,
    provincia,
    distrito
)
VALUES (
    -1,
    '000000',
    'No Especificado',
    'No Especificado',
    'No Especificado'
);

SET IDENTITY_INSERT dmt.DIM_UBICACION OFF;

SET IDENTITY_INSERT dmt.DIM_ENTIDAD ON;

INSERT INTO dmt.DIM_ENTIDAD (
    id_entidad,
    SEC_EJEC,
    EJECUTORA_NOMBRE,
    NIVEL_GOBIERNO_NOMBRE
)
VALUES (
    -1,
    '0000',
    'No Especificado',
    'No Especificado'
);

SET IDENTITY_INSERT dmt.DIM_ENTIDAD OFF;

SET IDENTITY_INSERT dmt.DIM_FINANCIERA ON;

INSERT INTO dmt.DIM_FINANCIERA (
    id_financiera,
    FUENTE_FINANCIAMIENTO_NOMBRE,
    RUBRO_NOMBRE,
    GENERICA_NOMBRE,
    SUBGENERICA_NOMBRE
)
VALUES (
    -1,
    'No Especificado',
    'No Especificado',
    'No Especificado',
    'No Especificado'
);

SET IDENTITY_INSERT dmt.DIM_FINANCIERA OFF;
GO

CREATE TABLE dmt.FACT_PRESUPUESTO (

    id_fact_presupuesto BIGINT IDENTITY(1,1),

    id_tiempo INT NOT NULL,
    id_ubicacion INT NOT NULL,
    id_entidad INT NOT NULL,
    id_financiera INT NOT NULL,

    MONTO_PIA DECIMAL(38,2),
    MONTO_PIM DECIMAL(38,2),
    MONTO_RECAUDADO DECIMAL(38,2),
    TASA_EJECUCION FLOAT,

    CONSTRAINT PK_FACT_PRESUPUESTO
    PRIMARY KEY NONCLUSTERED (id_fact_presupuesto),

    CONSTRAINT FK_FACT_TIEMPO
        FOREIGN KEY (id_tiempo)
        REFERENCES dmt.DIM_TIEMPO(id_tiempo),

    CONSTRAINT FK_FACT_UBICACION
        FOREIGN KEY (id_ubicacion)
        REFERENCES dmt.DIM_UBICACION(id_ubicacion),

    CONSTRAINT FK_FACT_ENTIDAD
        FOREIGN KEY (id_entidad)
        REFERENCES dmt.DIM_ENTIDAD(id_entidad),

    CONSTRAINT FK_FACT_FINANCIERA
        FOREIGN KEY (id_financiera)
        REFERENCES dmt.DIM_FINANCIERA(id_financiera)
);
GO

CREATE CLUSTERED COLUMNSTORE INDEX IX_FACT_PRESUPUESTO_COLUMNSTORE
ON dmt.FACT_PRESUPUESTO;
GO

ALTER TABLE dmt.DIM_TIEMPO
ADD CONSTRAINT CHK_MES
CHECK (mes = 0 OR (mes BETWEEN 1 AND 12));
GO

ALTER TABLE dmt.FACT_PRESUPUESTO
ADD CONSTRAINT CHK_MONTO_PIA
CHECK (MONTO_PIA >= 0);
GO

ALTER TABLE dmt.FACT_PRESUPUESTO
ADD CONSTRAINT CHK_MONTO_PIM
CHECK (MONTO_PIM >= 0);
GO

ALTER TABLE dmt.FACT_PRESUPUESTO
ADD CONSTRAINT CHK_MONTO_RECAUDADO
CHECK (MONTO_RECAUDADO >= 0);
GO
```
</details>

---

# FASE 4 — Desarrollo del ETL hacia SQL Server

En esta fase se desarrolló el proceso ETL encargado de cargar la información desde la capa Gold hacia el Data Mart en SQL Server. El flujo incluyó lectura del archivo Parquet, normalización de datos, poblamiento de dimensiones, uso de tablas staging y carga final hacia la tabla de hechos.

## Flujo General del ETL

| Etapa | Descripción |
|---|---|
| Lectura Gold | Carga del archivo Parquet |
| Normalización | Limpieza y conversión de tipos |
| Poblamiento dimensional | Inserción en dimensiones |
| Staging | Tabla temporal STG_PRESUPUESTO |
| Resolución de llaves | JOINs automáticos |
| Inserción final | FACT_PRESUPUESTO |


## Componentes Técnicos del ETL

| Componente | Tecnología |
|---|---|
| Lenguaje | Python |
| Librería ETL | Pandas |
| Conector SQL | SQLAlchemy |
| Driver SQL Server | pyodbc |
| Optimización | fast_executemany=True |



## Script Python ETL

<details>
<summary><strong>Ver Script SQL Completo</strong></summary>


```python
import pandas as pd
from sqlalchemy import create_engine, text
import urllib

SERVER = 'localhost'
DATABASE = 'DMT_PRESUPUESTO'

CONNECTION_STRING = (
    f"mssql+pyodbc://@{SERVER}/{DATABASE}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(CONNECTION_STRING, fast_executemany=True)

print("=" * 60)
print("      1. CARGA Y LIMPIEZA DEL DATASET GOLD")
print("=" * 60)

df = pd.read_parquet('gold_municipalidades.parquet')

print(f"Registros totales detectados : {len(df):,}")

columnas_sismepre = [
    'SISMEPRE_DECIMAL_PROM',
    'SISMEPRE_ENTERO_PROM',
    'SISMEPRE_ESTADO_MODA',
    'SISMEPRE_RESPUESTAS'
]

df = df.drop(
    columns=[c for c in columnas_sismepre if c in df.columns],
    errors='ignore'
)

print("Normalizando textos y mitigando registros nulos...")

columnas_texto = [
    'UBIGEO',
    'Departamento',
    'Provincia',
    'Distrito',
    'SEC_EJEC',
    'EJECUTORA_NOMBRE',
    'NIVEL_GOBIERNO',
    'NIVEL_GOBIERNO_NOMBRE',
    'DEPARTAMENTO_EJECUTORA_NOMBRE',
    'PROVINCIA_EJECUTORA_NOMBRE',
    'DISTRITO_EJECUTORA_NOMBRE',
    'FUENTE_FINANCIAMIENTO',
    'FUENTE_FINANCIAMIENTO_NOMBRE',
    'RUBRO',
    'RUBRO_NOMBRE',
    'GENERICA',
    'GENERICA_NOMBRE',
    'SUBGENERICA',
    'SUBGENERICA_NOMBRE'
]

for col in columnas_texto:
    if col in df.columns:
        df[col] = (
            df[col]
            .fillna('No Especificado')
            .astype(str)
            .str.strip()
        )

df['EJECUTORA'] = (
    pd.to_numeric(df['EJECUTORA'], errors='coerce')
    .fillna(0)
    .astype(int)
)

df['Tipomuni'] = (
    pd.to_numeric(df['Tipomuni'], errors='coerce')
    .fillna(0.0)
    .astype(float)
)

df['MONTO_PIA'] = (
    pd.to_numeric(df['MONTO_PIA'], errors='coerce')
    .fillna(0.0)
    .astype(float)
)

df['MONTO_PIM'] = (
    pd.to_numeric(df['MONTO_PIM'], errors='coerce')
    .fillna(0.0)
    .astype(float)
)

df['MONTO_RECAUDADO'] = (
    pd.to_numeric(df['MONTO_RECAUDADO'], errors='coerce')
    .fillna(0.0)
    .astype(float)
)

df['TASA_EJECUCION'] = (
    pd.to_numeric(df['TASA_EJECUCION'], errors='coerce')
    .fillna(0.0)
    .astype(float)
)

df['fecha_inicio_mes'] = pd.to_datetime(
    df['ANO_DOC'].astype(str) + '-' +
    df['MES_DOC'].astype(str) + '-01',
    errors='coerce'
)

print("\n" + "=" * 60)
print("      2. REINICIANDO DESTINOS EN SQL SERVER")
print("=" * 60)

with engine.begin() as conn:
    conn.execute(text("DELETE FROM dmt.FACT_PRESUPUESTO"))
    conn.execute(text("DELETE FROM dmt.DIM_FINANCIERA"))
    conn.execute(text("DELETE FROM dmt.DIM_ENTIDAD"))
    conn.execute(text("DELETE FROM dmt.DIM_UBICACION"))
    conn.execute(text("DELETE FROM dmt.DIM_TIEMPO"))

print("Base de datos lista para recibir la carga.")

print("\n" + "=" * 60)
print("      3. POBLANDO TABLAS DE DIMENSIONES")
print("=" * 60)

dim_tiempo = (
    df[['fecha_inicio_mes', 'ANO_DOC', 'MES_DOC']]
    .drop_duplicates()
    .dropna()
)

dim_tiempo['nombre_mes'] = (
    dim_tiempo['fecha_inicio_mes']
    .dt.month_name(locale='Spanish')
)

dim_tiempo['nombre_corto_mes'] = (
    dim_tiempo['nombre_mes'].str[:3]
)

dim_tiempo['trimestre'] = (
    'T' + (((dim_tiempo['MES_DOC'] - 1) // 3) + 1).astype(str)
)

dim_tiempo['semestre'] = (
    'S' + (((dim_tiempo['MES_DOC'] - 1) // 6) + 1).astype(str)
)

dim_tiempo['bimestre'] = (
    'B' + (((dim_tiempo['MES_DOC'] - 1) // 2) + 1).astype(str)
)

dim_tiempo.columns = [
    'fecha_inicio_mes',
    'anio',
    'mes',
    'nombre_mes',
    'nombre_corto_mes',
    'trimestre',
    'semestre',
    'bimestre'
]

dim_tiempo.to_sql(
    'DIM_TIEMPO',
    engine,
    schema='dmt',
    if_exists='append',
    index=False
)

print(f"dmt.DIM_TIEMPO poblada : {len(dim_tiempo):,}")

dim_ubicacion = df[
    ['UBIGEO', 'Departamento', 'Provincia', 'Distrito']
].copy()

dim_ubicacion = (
    dim_ubicacion[dim_ubicacion['UBIGEO'] != '']
    .drop_duplicates(subset=['UBIGEO'])
)

dim_ubicacion.columns = [
    'UBIGEO',
    'departamento',
    'provincia',
    'distrito'
]

dim_ubicacion.to_sql(
    'DIM_UBICACION',
    engine,
    schema='dmt',
    if_exists='append',
    index=False
)

print(f"dmt.DIM_UBICACION poblada : {len(dim_ubicacion):,}")

dim_entidad = df[[
    'SEC_EJEC',
    'EJECUTORA',
    'EJECUTORA_NOMBRE',
    'NIVEL_GOBIERNO',
    'NIVEL_GOBIERNO_NOMBRE',
    'DEPARTAMENTO_EJECUTORA_NOMBRE',
    'PROVINCIA_EJECUTORA_NOMBRE',
    'DISTRITO_EJECUTORA_NOMBRE',
    'Tipomuni'
]].copy()

dim_entidad = (
    dim_entidad[dim_entidad['SEC_EJEC'] != '']
    .drop_duplicates(subset=['SEC_EJEC'])
)

dim_entidad.to_sql(
    'DIM_ENTIDAD',
    engine,
    schema='dmt',
    if_exists='append',
    index=False
)

print(f"dmt.DIM_ENTIDAD poblada : {len(dim_entidad):,}")

dim_financiera = df[[
    'FUENTE_FINANCIAMIENTO',
    'FUENTE_FINANCIAMIENTO_NOMBRE',
    'RUBRO',
    'RUBRO_NOMBRE',
    'GENERICA',
    'GENERICA_NOMBRE',
    'SUBGENERICA',
    'SUBGENERICA_NOMBRE'
]].drop_duplicates(
    subset=[
        'FUENTE_FINANCIAMIENTO',
        'RUBRO',
        'GENERICA',
        'SUBGENERICA'
    ]
)

dim_financiera.to_sql(
    'DIM_FINANCIERA',
    engine,
    schema='dmt',
    if_exists='append',
    index=False
)

print(f"dmt.DIM_FINANCIERA poblada : {len(dim_financiera):,}")

print("\n" + "=" * 60)
print("      4. EJECUTANDO FASE DE STAGING EN SQL SERVER")
print("=" * 60)

print("Subiendo registros a dmt.STG_PRESUPUESTO...")

df_staging = df[[
    'fecha_inicio_mes',
    'UBIGEO',
    'SEC_EJEC',
    'FUENTE_FINANCIAMIENTO',
    'RUBRO',
    'GENERICA',
    'SUBGENERICA',
    'MONTO_PIA',
    'MONTO_PIM',
    'MONTO_RECAUDADO',
    'TASA_EJECUCION'
]]

df_staging.to_sql(
    'STG_PRESUPUESTO',
    con=engine,
    schema='dmt',
    if_exists='replace',
    index=False,
    chunksize=70000
)

print("Carga cruda completada en la tabla Staging.")

print("\n" + "=" * 60)
print("      5. RESOLVIENDO LLAVES DENTRO DE SQL SERVER")
print("=" * 60)

query_fact = """
INSERT INTO dmt.FACT_PRESUPUESTO (
    id_tiempo,
    id_ubicacion,
    id_entidad,
    id_financiera,
    MONTO_PIA,
    MONTO_PIM,
    MONTO_RECAUDADO,
    TASA_EJECUCION
)
SELECT
    ISNULL(t.id_tiempo, -1) AS id_tiempo,
    ISNULL(u.id_ubicacion, -1) AS id_ubicacion,
    ISNULL(e.id_entidad, -1) AS id_entidad,
    ISNULL(f.id_financiera, -1) AS id_financiera,
    s.MONTO_PIA,
    s.MONTO_PIM,
    s.MONTO_RECAUDADO,
    s.TASA_EJECUCION
FROM dmt.STG_PRESUPUESTO s
LEFT JOIN dmt.DIM_TIEMPO t
    ON s.fecha_inicio_mes = t.fecha_inicio_mes
LEFT JOIN dmt.DIM_UBICACION u
    ON s.UBIGEO = u.UBIGEO
LEFT JOIN dmt.DIM_ENTIDAD e
    ON s.SEC_EJEC = e.SEC_EJEC
LEFT JOIN dmt.DIM_FINANCIERA f
    ON s.FUENTE_FINANCIAMIENTO = f.FUENTE_FINANCIAMIENTO
    AND s.RUBRO = f.RUBRO
    AND s.GENERICA = f.GENERICA
    AND s.SUBGENERICA = f.SUBGENERICA;
"""

with engine.begin() as conn:
    conn.execute(text(query_fact))
    print("Cruce exitoso. Eliminando tabla STG_PRESUPUESTO...")
    conn.execute(text("DROP TABLE dmt.STG_PRESUPUESTO;"))

print("\n" + "=" * 60)
print("      ETL DATA MART COMPLETADO")
print("=" * 60)
```

</details>

---

# FASE 5 — Integración Analítica en Power BI

En esta fase se conectó el Data Mart implementado en SQL Server con Power BI para la construcción de dashboards analíticos. Se desarrollaron visualizaciones orientadas al análisis financiero municipal, evaluación presupuestal y monitoreo de indicadores de ejecución.

## Dashboards Desarrollados

| Dashboard | Objetivo |
|---|---|
| Resumen Ejecutivo Nacional | Visión global financiera |
| Análisis Territorial Municipal | Distribución geográfica |
| Eficiencia Institucional Municipal | Comparación municipal |
| Ingresos y Financiamiento | Estructura financiera |
| Tendencias y Estacionalidad | Patrones temporales |
| Riesgo y Alertas Presupuestales | Detección de ineficiencias |

---

## Resultado Final

El Data Mart Presupuestal Municipal permite realizar análisis multidimensionales de alto rendimiento sobre la ejecución presupuestal de municipalidades peruanas, integrando información temporal, geográfica, institucional y financiera en un entorno optimizado para Business Intelligence y visualización analítica en Power BI.